# 04 - Tracking Evaluation ByteTrack

Obiettivo:
- valutare la baseline RF-DETR + ByteTrack;
- confrontarla con BoT-SORT + ReID;
- calcolare HOTA, DetA, AssA, IDF1, MOTA, IDSW e Frag.

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path(
    "/Users/marcodalbis/Desktop/Tesi Magistrale/"
    "basketball-thesis/basketball-thesis"
)

TRACKEVAL_ROOT = PROJECT_ROOT / "TrackEval"

print("TrackEval root:", TRACKEVAL_ROOT)
print("Exists:", TRACKEVAL_ROOT.exists())

TrackEval root: /Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval
Exists: True


In [2]:
from pathlib import Path
import pandas as pd
import shutil
import os

DRIVE_ROOT = Path(
    "/Users/marcodalbis/Library/CloudStorage/"
    "GoogleDrive-marco.dalbis@gmail.com/"
    "Il mio Drive/"
    "basketball-thesis"
)

SPORTSMOT_ROOT = (
    DRIVE_ROOT
    / "datasets"
    / "sportsmot"
)

TRACKING_OUTPUT_ROOT = (
    DRIVE_ROOT
    / "experiments"
    / "bytetrack_baseline"
)

print("SportsMOT:", SPORTSMOT_ROOT)
print("Tracking outputs:", TRACKING_OUTPUT_ROOT)
print("TrackEval:", TRACKEVAL_ROOT)

SportsMOT: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/datasets/sportsmot
Tracking outputs: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/experiments/bytetrack_baseline
TrackEval: /Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval


Sequenza da analizzare

In [3]:
SEQUENCE_NAME = "v_00HRwkvvjtQ_c001"

GT_SEQUENCE_DIR = (
    SPORTSMOT_ROOT
    / "val"
    / SEQUENCE_NAME
)

GT_FILE = GT_SEQUENCE_DIR / "gt" / "gt.txt"

PREDICTIONS_CSV = (
    TRACKING_OUTPUT_ROOT
    / f"{SEQUENCE_NAME}_tracks.csv"
)

print("GT:", GT_FILE)
print("Predizioni:", PREDICTIONS_CSV)

print("GT esiste:", GT_FILE.exists())
print("Predizioni esistono:", PREDICTIONS_CSV.exists())

GT: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/datasets/sportsmot/val/v_00HRwkvvjtQ_c001/gt/gt.txt
Predizioni: /Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/experiments/bytetrack_baseline/v_00HRwkvvjtQ_c001_tracks.csv
GT esiste: True
Predizioni esistono: True


In [4]:
gt_df = pd.read_csv(
    GT_FILE,
    header=None
)

print("Shape GT:", gt_df.shape)

gt_df.head()

Shape GT: (11276, 9)


,0,1,2,3,4,5,6,7,8
0,1,0,438,269,118,169,1,1,1
1,1,1,973,389,78,216,1,1,1
2,1,2,634,429,95,198,1,1,1
3,1,3,649,272,108,152,1,1,1
4,1,4,637,421,129,159,1,1,1


In [5]:
print("Numero frame GT:", gt_df[0].nunique())
print("Numero ID GT:", gt_df[1].nunique())

print("Frame minimo:", gt_df[0].min())
print("Frame massimo:", gt_df[0].max())

Numero frame GT: 1162
Numero ID GT: 10
Frame minimo: 1
Frame massimo: 1162


In [6]:
pred_df = pd.read_csv(PREDICTIONS_CSV)

print(pred_df.shape)
pred_df.head()

(10911, 8)


,frame,time,track_id,confidence,x1,y1,x2,y2
0,2,0.04,0,0.946324,434.150208,268.251038,560.730225,436.969086
1,2,0.04,1,0.918907,973.851013,390.865112,1050.417969,604.696655
2,2,0.04,2,0.847887,636.885437,416.111542,768.735291,625.620422
3,2,0.04,3,0.852696,990.499023,290.476715,1053.694336,497.539978
4,2,0.04,4,0.760175,652.945068,272.642029,758.719971,433.214264


In [7]:
print("Predizioni:")
print("Frame minimo:", pred_df["frame"].min())
print("Frame massimo:", pred_df["frame"].max())
print("Numero frame:", pred_df["frame"].nunique())

print()

print("Ground truth:")
print("Frame minimo:", gt_df[0].min())
print("Frame massimo:", gt_df[0].max())
print("Numero frame:", gt_df[0].nunique())

Predizioni:
Frame minimo: 2
Frame massimo: 1162
Numero frame: 1161

Ground truth:
Frame minimo: 1
Frame massimo: 1162
Numero frame: 1162


## Controllo tracker

In [8]:
print(
    "Tracker ID invalidi (-1):",
    (pred_df["track_id"] == -1).sum()
)

print(
    "Tracker ID unici:",
    pred_df["track_id"].nunique()
)

Tracker ID invalidi (-1): 0
Tracker ID unici: 27


Conversione delle bbox nel formato MOTChallenge

In [9]:
mot_pred = pd.DataFrame()

mot_pred["frame"] = pred_df["frame"].astype(int)
mot_pred["id"] = pred_df["track_id"].astype(int)

mot_pred["bb_left"] = pred_df["x1"]
mot_pred["bb_top"] = pred_df["y1"]

mot_pred["bb_width"] = (
    pred_df["x2"] - pred_df["x1"]
)

mot_pred["bb_height"] = (
    pred_df["y2"] - pred_df["y1"]
)

mot_pred["conf"] = pred_df["confidence"]

mot_pred["x"] = -1
mot_pred["y"] = -1
mot_pred["z"] = -1

mot_pred.head()

,frame,id,bb_left,bb_top,bb_width,bb_height,conf,x,y,z
0,2,0,434.150208,268.251038,126.580017,168.718048,0.946324,-1,-1,-1
1,2,1,973.851013,390.865112,76.566956,213.831543,0.918907,-1,-1,-1
2,2,2,636.885437,416.111542,131.849854,209.508881,0.847887,-1,-1,-1
3,2,3,990.499023,290.476715,63.195312,207.063263,0.852696,-1,-1,-1
4,2,4,652.945068,272.642029,105.774902,160.572235,0.760175,-1,-1,-1


In [10]:
# Ordinamento MOTChallenge
mot_pred = mot_pred.sort_values(
    ["frame", "id"]
).reset_index(drop=True)

# Controlli di consistenza
assert mot_pred.shape[1] == 10, (
    f"Numero colonne errato: {mot_pred.shape[1]}"
)

assert not mot_pred.isna().any().any(), (
    "Sono presenti valori NaN nelle predizioni."
)

assert (mot_pred["frame"] >= 1).all()
assert (mot_pred["id"] >= 0).all()

print("Numero righe:", len(mot_pred))
print("Numero colonne:", mot_pred.shape[1])
print("Frame:", mot_pred["frame"].min(), "→", mot_pred["frame"].max())
print("Track ID unici:", mot_pred["id"].nunique())

mot_pred.head()

Numero righe: 10911
Numero colonne: 10
Frame: 2 → 1162
Track ID unici: 27


,frame,id,bb_left,bb_top,bb_width,bb_height,conf,x,y,z
0,2,0,434.150208,268.251038,126.580017,168.718048,0.946324,-1,-1,-1
1,2,1,973.851013,390.865112,76.566956,213.831543,0.918907,-1,-1,-1
2,2,2,636.885437,416.111542,131.849854,209.508881,0.847887,-1,-1,-1
3,2,3,990.499023,290.476715,63.195312,207.063263,0.852696,-1,-1,-1
4,2,4,652.945068,272.642029,105.774902,160.572235,0.760175,-1,-1,-1


Controllo bbox invalide

In [11]:
invalid_boxes = mot_pred[
    (mot_pred["bb_width"] <= 0)
    | (mot_pred["bb_height"] <= 0)
    | mot_pred["bb_left"].isna()
    | mot_pred["bb_top"].isna()
    | mot_pred["conf"].isna()
]

print("Bounding box invalide:", len(invalid_boxes))

invalid_boxes.head()

Bounding box invalide: 0


,frame,id,bb_left,bb_top,bb_width,bb_height,conf,x,y,z


In [12]:
assert len(invalid_boxes) == 0, (
    "Sono presenti bounding box invalide."
)

Preparazione struttura TrackEval

In [13]:
BENCHMARK = "SportsMOT"
SPLIT = "val"
TRACKER_NAME = "ByteTrack"

GT_ROOT = (
    TRACKEVAL_ROOT
    / "data"
    / "gt"
    / "mot_challenge"
    / f"{BENCHMARK}-{SPLIT}"
)

TRACKER_ROOT = (
    TRACKEVAL_ROOT
    / "data"
    / "trackers"
    / "mot_challenge"
    / f"{BENCHMARK}-{SPLIT}"
    / TRACKER_NAME
    / "data"
)

GT_ROOT.mkdir(parents=True, exist_ok=True)
TRACKER_ROOT.mkdir(parents=True, exist_ok=True)

print(GT_ROOT)
print(TRACKER_ROOT)

/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/gt/mot_challenge/SportsMOT-val
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/ByteTrack/data


## Copia del ground truth

sequence/
├── gt/
│   └── gt.txt
└── seqinfo.ini

In [14]:
target_sequence_dir = GT_ROOT / SEQUENCE_NAME
target_gt_dir = target_sequence_dir / "gt"

target_gt_dir.mkdir(parents=True, exist_ok=True)

shutil.copy(
    GT_SEQUENCE_DIR / "gt" / "gt.txt",
    target_gt_dir / "gt.txt"
)

shutil.copy(
    GT_SEQUENCE_DIR / "seqinfo.ini",
    target_sequence_dir / "seqinfo.ini"
)

print("GT copiato.")

GT copiato.


In [15]:
TRACKER_FILE = TRACKER_ROOT / f"{SEQUENCE_NAME}.txt"

mot_pred.to_csv(
    TRACKER_FILE,
    index=False,
    header=False,
)

print("Predizioni salvate in:")
print(TRACKER_FILE)

Predizioni salvate in:
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/ByteTrack/data/v_00HRwkvvjtQ_c001.txt


creazione seqmap

In [16]:
SEQMAP_DIR = (
    TRACKEVAL_ROOT
    / "data"
    / "gt"
    / "mot_challenge"
    / "seqmaps"
)

SEQMAP_DIR.mkdir(parents=True, exist_ok=True)

SEQMAP_FILE = SEQMAP_DIR / f"{BENCHMARK}-{SPLIT}.txt"

with open(SEQMAP_FILE, "w") as f:
    f.write("name\n")
    f.write(f"{SEQUENCE_NAME}\n")

print(SEQMAP_FILE.read_text())

name
v_00HRwkvvjtQ_c001



controllo sequenza finale

In [17]:
print("Ground truth:")
print(GT_ROOT)

print("\nTracker:")
print(TRACKER_ROOT)

print("\nSeqmap:")
print(SEQMAP_FILE)

print("\nPredizione:")
print(TRACKER_FILE)

Ground truth:
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/gt/mot_challenge/SportsMOT-val

Tracker:
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/ByteTrack/data

Seqmap:
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/gt/mot_challenge/seqmaps/SportsMOT-val.txt

Predizione:
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/ByteTrack/data/v_00HRwkvvjtQ_c001.txt


In [18]:
for path in (
    TRACKEVAL_ROOT
    / "data"
    / "gt"
    / "mot_challenge"
    / "SportsMOT-val"
).rglob("*"):
    if path.is_file():
        print(path)

/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/gt/mot_challenge/SportsMOT-val/v_00HRwkvvjtQ_c001/seqinfo.ini
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/gt/mot_challenge/SportsMOT-val/v_00HRwkvvjtQ_c001/gt/gt.txt


In [19]:
for path in (
    TRACKEVAL_ROOT
    / "data"
    / "trackers"
    / "mot_challenge"
    / "SportsMOT-val"
).rglob("*"):
    if path.is_file():
        print(path)

/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/ByteTrack/data/v_00HRwkvvjtQ_c001.txt
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/BoTSORT_ReID/pedestrian_summary.txt
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/BoTSORT_ReID/pedestrian_detailed.csv
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/BoTSORT_ReID/pedestrian_plot.pdf
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/BoTSORT_ReID/pedestrian_plot.png
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/BoTSORT_ReID/data/v_00HRwkvvjtQ_c001.

## Prima valutazione TrackEval

In [20]:
import subprocess
import sys

cmd = [
    sys.executable,
    "scripts/run_mot_challenge.py",
    "--BENCHMARK", "SportsMOT",
    "--SPLIT_TO_EVAL", "val",
    "--TRACKERS_TO_EVAL", TRACKER_NAME,
    "--METRICS", "HOTA", "CLEAR", "Identity",
    "--USE_PARALLEL", "False",
    "--DO_PREPROC", "False",
]

result = subprocess.run(
    cmd,
    cwd=TRACKEVAL_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

print("Return code:", result.returncode)

Error importing BURST due to missing underlying dependency: No module named 'pycocotools'

Eval Config:
USE_PARALLEL         : False                         
NUM_PARALLEL_CORES   : 8                             
BREAK_ON_ERROR       : True                          
RETURN_ON_ERROR      : False                         
LOG_ON_ERROR         : /Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/error_log.txt
PRINT_RESULTS        : True                          
PRINT_ONLY_COMBINED  : False                         
PRINT_CONFIG         : True                          
TIME_PROGRESS        : True                          
DISPLAY_LESS_PROGRESS : False                         
OUTPUT_SUMMARY       : True                          
OUTPUT_EMPTY_CLASSES : True                          
OUTPUT_DETAILED      : True                          
PLOT_CURVES          : True                          

MotChallenge2DBox Config:
PRINT_CONFIG         : True             

## Interpretazione delle metriche

Le principali metriche considerate sono:

### HOTA
Misura congiuntamente:
- qualità della detection;
- qualità dell'associazione temporale.

### DetA
Misura la componente di detection.

### AssA
Misura quanto bene viene mantenuta l'identità dei giocatori nel tempo.

### IDF1
Misura la correttezza delle identità lungo le traiettorie.

### MOTA
Combina:
- false positive;
- false negative;
- ID switches.

### IDSW
Numero di cambi errati di identità.

### Frag
Numero di frammentazioni delle tracce.

Individuazione file risultato

In [21]:
result_files = list(
    (
        TRACKEVAL_ROOT
        / "data"
        / "trackers"
        / "mot_challenge"
        / f"{BENCHMARK}-{SPLIT}"
        / TRACKER_NAME
    ).rglob("*")
)

for f in result_files:
    if f.is_file():
        print(f)

/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/ByteTrack/pedestrian_summary.txt
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/ByteTrack/pedestrian_detailed.csv
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/ByteTrack/pedestrian_plot.pdf
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/ByteTrack/pedestrian_plot.png
/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/ByteTrack/data/v_00HRwkvvjtQ_c001.txt


Lettura summary

In [22]:
summary_files = [
    f for f in result_files
    if "summary" in f.name.lower()
]

summary_files

[PosixPath('/Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/ByteTrack/pedestrian_summary.txt')]

In [23]:
for summary_file in summary_files:
    print("\nFILE:", summary_file)
    print(summary_file.read_text())


FILE: /Users/marcodalbis/Desktop/Tesi Magistrale/basketball-thesis/basketball-thesis/TrackEval/data/trackers/mot_challenge/SportsMOT-val/ByteTrack/pedestrian_summary.txt
HOTA DetA AssA DetRe DetPr AssRe AssPr LocA OWTA HOTA(0) LocA(0) HOTALocA(0) MOTA MOTP MODA CLR_Re CLR_Pr MTR PTR MLR CLR_TP CLR_FN CLR_FP IDSW MT PT ML Frag sMOTA IDF1 IDR IDP IDTP IDFN IDFP Dets GT_Dets IDs GT_IDs
52.445 78.203 35.231 82.778 85.547 41.826 64.766 86.623 54.001 63.203 84.087 53.146 94.032 84.852 94.475 95.619 98.818 100 0 0 10782 494 129 50 10 0 0 131 79.548 58.773 57.822 59.756 6520 4756 4391 10911 11276 27 10



In [24]:
summary_tables = []

for summary_file in summary_files:

    try:
        df_summary = pd.read_csv(
            summary_file,
            sep=r"\s+"
        )

        df_summary["source_file"] = summary_file.name

        summary_tables.append(df_summary)

    except Exception as e:
        print(
            "Errore leggendo",
            summary_file,
            ":",
            e
        )

Salvataggio risultati sul Drive

In [25]:
EVAL_OUTPUT_DIR = (
    DRIVE_ROOT
    / "experiments"
    / "bytetrack_baseline"
    / "evaluation"
    / SEQUENCE_NAME
)

EVAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TRACKER_RESULT_DIR = (
    TRACKEVAL_ROOT
    / "data"
    / "trackers"
    / "mot_challenge"
    / f"{BENCHMARK}-{SPLIT}"
    / TRACKER_NAME
)

shutil.copytree(
    TRACKER_RESULT_DIR,
    EVAL_OUTPUT_DIR,
    dirs_exist_ok=True,
)

print("Risultati copiati in:")
print(EVAL_OUTPUT_DIR)

Risultati copiati in:
/Users/marcodalbis/Library/CloudStorage/GoogleDrive-marco.dalbis@gmail.com/Il mio Drive/basketball-thesis/experiments/bytetrack_baseline/evaluation/v_00HRwkvvjtQ_c001


In [26]:
if summary_tables:

    tracking_metrics_df = pd.concat(
        summary_tables,
        ignore_index=True
    )

    display(tracking_metrics_df)

else:
    tracking_metrics_df = pd.DataFrame()

    print("Nessun summary leggibile trovato.")

,HOTA,DetA,AssA,DetRe,DetPr,AssRe,AssPr,LocA,OWTA,HOTA(0),...,IDR,IDP,IDTP,IDFN,IDFP,Dets,GT_Dets,IDs,GT_IDs,source_file
0,52.445,78.203,35.231,82.778,85.547,41.826,64.766,86.623,54.001,63.203,...,57.822,59.756,6520,4756,4391,10911,11276,27,10,pedestrian_summary.txt


# Risultati Tracking Evaluation

Sequenza valutata:

`v_00HRwkvvjtQ_c001`

Pipeline:

RF-DETR Nano fine-tuned
→ player detection
→ ByteTrack
→ track ID persistenti  
→ TrackEval

Metriche principali:

- HOTA:
- DetA:
- AssA:
- LocA:
- IDF1:
- IDP:
- IDR:
- MOTA:
- MOTP:
- ID switches:
- Fragmentation:

## Interpretazione preliminare

Da compilare dopo l'esecuzione:

- qualità complessiva del tracking;
- differenza tra componente detection e association;
- stabilità degli ID;
- presenza di ID switch;
- frammentazione delle tracce;
- comportamento durante le occlusioni;
- eventuali errori dovuti al detector;
- confronto con la baseline RF-DETR + ByteTrack.

## Conclusione

I risultati ottenuti rappresentano la baseline RF-DETR + ByteTrack.

Saranno confrontati con quelli ottenuti dalla variante RF-DETR + BoT-SORT + ReID per verificare se l'integrazione dell'informazione di appearance tramite ReID migliora l'associazione temporale, riduce gli ID switch e aumenta la continuità delle identità nel tempo.